# plife_torch — GPU model for perturbation response

**Input:** one file, `plife_data.npz`, from `pack_for_kaggle.py`.
Needs only `genes`, `<ctx>_perts`, `<ctx>_delta`. No h5ad, no graph, no STRING.

### What this replaces
Four 3,000-epoch CPU runs established:
* the model must compute the query from **features**, not look it up — 1,393 of
  1,393 held-out target genes had never been trained as queries (cold start);
* propagation contributes **zero** — pooled over 89 paired evals across four
  graphs and two force laws, effect = [-0.00062, +0.00007];
* the surviving model is `pred = Feat[tgt] · B · Featᵀ`, a rank-d **bilinear
  form**, best 0.5689 vs a nearest-neighbour bar of 0.5265.

At F=256 the retrieval bar **rose** (0.5265→0.5354) while the model **fell**
(0.5689→0.5517). Richer features helped a dumb lookup and hurt us — direct
evidence of structure a bilinear form cannot express. Hence two new branches:

1. **MLP query map** — is bilinear the ceiling?
2. **Cross-attention over training perturbations** — learned, soft, multi-neighbour
   retrieval. Hard top-1 NN is the saturated special case, so this should
   dominate the bar by construction.

### Non-negotiables carried over
* split by **perturbation**; features from **fit rows only**
* **union target mask** — every knocked-down column zeroed in every vector
* **debiased rank** — raw rank's null is 0.62, not 0.5
* **shuffled-query control at every eval** — must stay ~0.50 or something leaked
* attention **must not see the training batch** — masking only self inverts the
  loss (leave-one-out bias); verified at random init, null 0.0000 vs 0.5270

## 0. Environment check — DO NOT pip install anything

`numpy`, `torch` and `matplotlib` all ship in Kaggle's GPU image and are the
only dependencies. **Do not `pip install torch`**: it replaces the CUDA-matched
build with a generic wheel and you lose the GPU. Same for `numpy` — a reinstall
forces an ABI rebuild that breaks torch's bindings mid-session.

Everything used here is long-stable API (`torch.linalg.qr` / `svd` since 1.8,
`index_fill` / `scatter` / `GELU` older still), so version drift is not a real
risk. This cell just proves it before you spend GPU minutes.

In [ ]:
import sys, platform
import numpy as _np, torch as _t
print(f"python      {platform.python_version()}")
print(f"numpy       {_np.__version__}")
print(f"torch       {_t.__version__}")
print(f"cuda build  {_t.version.cuda}")
print(f"gpu         {_t.cuda.get_device_name(0) if _t.cuda.is_available() else 'NONE'}")
if _t.cuda.is_available():
    free, tot = _t.cuda.mem_get_info()
    print(f"vram        {free/1e9:.1f} GB free / {tot/1e9:.1f} GB total")

assert _t.cuda.is_available(), "No GPU. Settings -> Accelerator -> GPU, then restart."
assert tuple(int(x) for x in _t.__version__.split(".")[:2]) >= (1, 9), "need torch>=1.9"
for _f in ("linalg.qr", "linalg.svd", "cuda.mem_get_info"):
    _o = _t
    for _p in _f.split("."):
        _o = getattr(_o, _p, None)
        assert _o is not None, f"torch.{_f} missing — unexpected, tell me the version above"
# 708 MB for the K562 matrix in fp32, plus model and activations
assert _t.cuda.mem_get_info()[0] > 4e9, "under 4 GB VRAM free; restart the session"
print("\nenvironment OK — no installs needed")

In [ ]:
import math, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False})

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "|", DEV,
      torch.cuda.get_device_name(0) if DEV == "cuda" else "")

CFG = dict(
    path      = "/kaggle/input/datasets/ericdu847/plife-data-npz/plife_data.npz",
    ctx       = "k562",
    F         = 128,     # gene feature dim (SVD rank)
    D         = 96,      # bilinear branch width
    DK        = 128,     # attention key/query width
    hidden    = 512,     # MLP hidden width
    mode      = "both",  # 'bilinear' | 'attn' | 'both'
    steps     = 6000,
    batch     = 64,      # also the number of InfoNCE negatives
    temp      = 0.07,
    lr        = 2e-3,
    wd        = 1e-4,
    mag_w     = 0.1,     # weight on the magnitude head (0 disables)
    eval_every= 200,
    eval_chunk= 256,
    val_frac  = 0.15,
    seed      = 0,
)
torch.manual_seed(CFG["seed"])
print(json.dumps(CFG, indent=1))

## 1. Load and split by perturbation

In [ ]:
import os, glob

def resolve_npz(p):
    """Kaggle mounts a DATASET AS A DIRECTORY; the .npz lives inside it. Accept a
    file, a directory, or nothing at all, and say what was found either way."""
    if os.path.isfile(p):
        return p
    cands = []
    if os.path.isdir(p):
        cands += sorted(glob.glob(os.path.join(p, "**", "*.npz"), recursive=True))
    cands += sorted(glob.glob("/kaggle/input/**/plife_data.npz", recursive=True))
    cands += sorted(glob.glob("/kaggle/input/**/*.npz", recursive=True))
    seen = [c for i, c in enumerate(cands) if c not in cands[:i]]
    named = [c for c in seen if os.path.basename(c) == "plife_data.npz"]
    pick = (named or seen)
    if not pick:
        print("Nothing found. /kaggle/input contains:")
        for r, _d, fs in os.walk("/kaggle/input"):
            for f in fs:
                print("   ", os.path.join(r, f))
        raise FileNotFoundError(
            f"No .npz under {p!r} or /kaggle/input. Add the dataset via "
            "'+ Add Input', then set CFG['path'] to the printed path.")
    best = max(pick, key=os.path.getsize)
    if len(seen) > 1:
        print(f"note: {len(seen)} .npz files visible, using the largest match")
    return best

CFG["path"] = resolve_npz(CFG["path"])
print(f"loading {CFG['path']}  ({os.path.getsize(CFG['path'])/1e6:.0f} MB)")

z = np.load(CFG["path"], allow_pickle=True)
print("keys:", list(z.keys()))
_need = ["genes", f"{CFG['ctx']}_perts", f"{CFG['ctx']}_delta"]
_miss = [k for k in _need if k not in z]
assert not _miss, (f"missing {_miss}. The bundle has {list(z.keys())} — either "
                   f"re-run pack_for_kaggle.py or change CFG['ctx'].")

genes = [str(g) for g in z["genes"]]
gpos  = {g: i for i, g in enumerate(genes)}
G = len(genes)

perts = [str(p) for p in z[f"{CFG['ctx']}_perts"]]
keep  = np.array([i for i, p in enumerate(perts) if p in gpos])
Dnp   = z[f"{CFG['ctx']}_delta"][keep]              # stays fp16 on host
tgt   = np.array([gpos[perts[i]] for i in keep])

rng  = np.random.default_rng(CFG["seed"])
prm  = rng.permutation(len(keep))
ncut = int((1 - CFG["val_frac"]) * len(keep))
fit, val = np.sort(prm[:ncut]), np.sort(prm[ncut:])
print(f"{len(keep)} perturbations x {G} genes   fit {len(fit)}  val {len(val)}")

Dt   = torch.from_numpy(Dnp).to(DEV).float()        # RAW responses, never centred
tgt_t= torch.from_numpy(tgt).long().to(DEV)
fit_t= torch.from_numpy(fit).long().to(DEV)
val_t= torch.from_numpy(val).long().to(DEV)
print(f"Dt {tuple(Dt.shape)} {Dt.dtype}  {Dt.element_size()*Dt.nelement()/1e9:.2f} GB on {DEV}")

## 2. Gene features — the cold-start fix

`Feat[g]` = how gene *g* behaves **as a responder** across thousands of
perturbations. Defined for all 18,080 genes, including ones never perturbed.

Not leakage: the **label** for perturbation *t* is *t's row* (what happens when
you knock *t* down); the **feature** is *t's column* (how *t* moves when others
are knocked down). Different objects, and only fit rows are used.

In [ ]:
@torch.no_grad()
def build_features(Dt, fit_t, F):
    gmean = Dt.mean(0, keepdim=True)
    Xf = Dt[fit_t] - gmean                          # centre for geometry only
    R  = torch.randn(Dt.shape[1], 2 * F, device=Dt.device)
    Q, _ = torch.linalg.qr(Xf @ R)                  # (n_fit, 2F)
    B  = Q.T @ Xf                                   # (2F, G)
    _, S, Vh = torch.linalg.svd(B, full_matrices=False)
    Ft = (Vh[:F].T * S[:F]).contiguous()            # (G, F)
    del Xf, R, Q, B, Vh
    torch.cuda.empty_cache() if Dt.is_cuda else None
    return Ft / Ft.norm(dim=1, keepdim=True).clamp_min(1e-9)

Feat = build_features(Dt, fit_t, CFG["F"])
print("Feat", tuple(Feat.shape))

## 3. Metrics — identical to the CPU pipeline so numbers stay comparable

`rank_debiased` column-centres the similarity matrix. Without it the null is
0.618, not 0.5, because predictions from one model share a component and some
truths are liked by every prediction. Verified on a random-init model:
raw rank correct-pairing 0.6537 / random-pairing 0.6179; debiased 0.5418 / 0.4995.

In [ ]:
def prep(X, mask):
    a = X * mask
    a = a - a.mean(1, keepdim=True)
    return a / a.norm(dim=1, keepdim=True).clamp_min(1e-12)

def _ranks(S):
    n = S.shape[0]
    d = S.diagonal().unsqueeze(1)
    return 1.0 - (S > d).sum(1).float() / max(n - 1, 1)

def rank_debiased(P, Y, mask, per_row=False):
    S = prep(P, mask) @ prep(Y, mask).T
    r = _ranks(S - S.mean(0, keepdim=True))
    return r if per_row else r.mean().item()

def rank_raw(P, Y, mask):
    return _ranks(prep(P, mask) @ prep(Y, mask).T).mean().item()

def cos_rows(P, Y, mask):
    return (prep(P, mask) * prep(Y, mask)).sum(1).mean().item()

def union_mask(idx_global):
    m = torch.ones(1, G, device=DEV)
    m[0, tgt_t[idx_global]] = 0.0
    return m

# sanity: the metric's null must be 0.5
with torch.no_grad():
    _m = union_mask(val_t); _Y = Dt[val_t]
    _noise = torch.randn_like(_Y)
    _const = _Y.mean(0, keepdim=True).expand_as(_Y)
    print(f"null check  noise {rank_debiased(_noise,_Y,_m):.4f}   "
          f"constant {rank_debiased(_const,_Y,_m):.4f}   (both must be ~0.500)")

## 4. Baselines — the bar the model has to clear

In [ ]:
with torch.no_grad():
    Mv, Yv = union_mask(val_t), Dt[val_t]
    nn_src = fit_t[(Feat[tgt_t[val_t]] @ Feat[tgt_t[fit_t]].T).argmax(1)]
    BASE = {
        "corpus mean response":        Dt[fit_t].mean(0, keepdim=True).expand_as(Yv),
        "a random OTHER perturbation": Dt[fit_t[torch.randint(len(fit_t), (len(val_t),), device=DEV)]],
        "NN retrieval on Feat  <-BAR": Dt[nn_src],
    }
    print(f"{'baseline':<32}{'cosine':>9}{'rank':>8}{'debiased':>10}")
    BAR = -9.0
    for k, P in BASE.items():
        c, rr, rd = cos_rows(P, Yv, Mv), rank_raw(P, Yv, Mv), rank_debiased(P, Yv, Mv)
        BAR = max(BAR, rd)
        print(f"{k:<32}{c:>9.4f}{rr:>8.4f}{rd:>10.4f}")
    print(f"{'>>> THE BAR':<32}{'':>9}{'':>8}{BAR:>10.4f}")

## 5. Model

**Bilinear branch** — `MLP_q(Feat[tgt]) · MLP_r(Feat)ᵀ`. The MLPs are the new
part; with `nn.Linear` only, this is exactly the CPU model.

**Attention branch** — soft retrieval over the fit perturbations. Values are the
actual fit responses, so the model composes observed responses rather than
synthesising from scratch.

> **`drop` is load-bearing, and it must remove the WHOLE BATCH — not just self.**
> During training every example sits in the attention pool. Masking only your own
> row makes `pred_i` a leave-one-out mean: it still contains every `Y_j` in the
> batch but not `Y_i`, so the diagonal is systematically *less* similar than the
> off-diagonal and the InfoNCE signal inverts. Measured on a random-init mock,
> null must be 0.500:
>
> ```
> exclude SELF only      0.0000   <- inverted
> exclude WHOLE BATCH    0.5270   <- symmetric, unbiased
> ```
>
> Masking the batch is symmetric across the diagonal, still makes the row's own
> response unreachable (max attention weight on any batch row = 0), and leaves
> the gate untouched since val rows are never in the pool.

In [ ]:
def mlp(i, h, o):
    return nn.Sequential(nn.Linear(i, h), nn.GELU(), nn.Linear(h, o))

class Plife(nn.Module):
    def __init__(self, Feat, Vfit, tfit, cfg):
        super().__init__()
        self.register_buffer("Feat", Feat)
        self.register_buffer("V", Vfit)         # (n_fit, G) attention values
        self.register_buffer("tfit", tfit)      # (n_fit,) target gene per pool row
        f, d, dk, h = Feat.shape[1], cfg["D"], cfg["DK"], cfg["hidden"]
        self.mode = cfg["mode"]
        self.q = mlp(f, h, d)
        self.r = mlp(f, h, d)
        self.kq = mlp(f, h, dk)
        self.kk = mlp(f, h, dk)
        self.scale = nn.Parameter(torch.tensor(0.0))
        self.mix = nn.Parameter(torch.zeros(2))
        self.mag = mlp(f, h, 1)
        self.dk = dk

    def forward(self, tgt_idx, drop=None, force_mode=None):
        mode = force_mode or self.mode
        ft = self.Feat[tgt_idx]
        out, parts = 0.0, []
        w = torch.softmax(self.mix, 0)
        if mode in ("bilinear", "both"):
            parts.append(("bi", self.q(ft) @ self.r(self.Feat).T))
        if mode in ("attn", "both"):
            s = (self.kq(ft) @ self.kk(self.Feat[self.tfit]).T) / math.sqrt(self.dk)
            s = s * self.scale.exp()
            if drop is not None:            # remove the WHOLE batch from the pool
                s = s.index_fill(1, drop, float("-inf"))
            parts.append(("at", torch.softmax(s, 1) @ self.V))
        if len(parts) == 1:
            out = parts[0][1]
        else:
            out = w[0] * parts[0][1] + w[1] * parts[1][1]
        return out, self.mag(ft).squeeze(1)

model = Plife(Feat, Dt[fit_t].contiguous(), tgt_t[fit_t].contiguous(), CFG).to(DEV)
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["steps"])
print(f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters")

## 6. Gate with controls — run at every eval, never read a number without them

In [ ]:
@torch.no_grad()
def gate(model):
    model.eval()
    out = {}
    shuf = val_t[torch.randperm(len(val_t), device=DEV)]
    jobs = [("FULL", val_t, None)]
    if CFG["mode"] == "both":
        jobs += [("bilinear only", val_t, "bilinear"), ("attention only", val_t, "attn")]
    jobs += [("query SHUFFLED (must be ~.50)", shuf, None)]
    for name, idx, fm in jobs:
        P = torch.empty(len(idx), G, device=DEV)
        for s in range(0, len(idx), CFG["eval_chunk"]):
            e = min(s + CFG["eval_chunk"], len(idx))
            P[s:e], _ = model(tgt_t[idx[s:e]], None, fm)   # val rows are not in the pool
        out[name] = (cos_rows(P, Yv, Mv), rank_raw(P, Yv, Mv),
                     rank_debiased(P, Yv, Mv), rank_debiased(P, Yv, Mv, per_row=True))
    model.train()
    return out

## 7. Train

In [ ]:
n_fit = len(fit_t)
best, best_step, hist = -9.0, -1, []
t0 = time.time()
model.train()
for step in range(CFG["steps"] + 1):
    bp = torch.randint(n_fit, (CFG["batch"],), device=DEV)   # positions in the pool
    gi = fit_t[bp]                                           # global row indices
    pred, magp = model(tgt_t[gi], drop=bp)         # mask the whole batch, not just self
    M = union_mask(gi)
    Y = Dt[gi]
    u, y = prep(pred, M), prep(Y, M)
    S = (u @ y.T) / CFG["temp"]
    loss = Fn.cross_entropy(S, torch.arange(len(bp), device=DEV))
    if CFG["mag_w"]:
        tgt_mag = (Y * M).abs().mean(1).clamp_min(1e-8).log()
        loss = loss + CFG["mag_w"] * Fn.mse_loss(magp, tgt_mag)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); sched.step()

    if step % CFG["eval_every"] == 0:
        g = gate(model)
        full = g["FULL"][2]; sh = g["query SHUFFLED (must be ~.50)"][2]
        hist.append((step, loss.item(), full, sh))
        flag = ""
        if full > best:
            best, best_step = full, step
            torch.save({"model": model.state_dict(), "cfg": CFG, "step": step,
                        "debiased": full}, "plife_best.pt")
            flag = " <- best"
        extra = ""
        if CFG["mode"] == "both":
            extra = f"  [bi {g['bilinear only'][2]:.4f}  attn {g['attention only'][2]:.4f}]"
        warn = "   ** SHUFFLED CONTROL HIGH — INVESTIGATE **" if sh > 0.53 else ""
        print(f"step {step:>6}  loss {loss.item():>6.3f}  debiased {full:.4f}  "
              f"bar {BAR:.4f}  shuf {sh:.4f}{extra}{flag}{warn}")
print(f"\nbest {best:.4f} @ step {best_step}   [{time.time()-t0:.0f}s]")

### Training curve
The gate is **not** the loss — on the CPU runs the loss kept falling while the
gate turned over. Read the gate. `shuffled` must stay flat at 0.50: if it rises
with the model, identity has leaked and every number above it is void.

In [ ]:
H = np.array(hist)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(H[:, 0], H[:, 1], lw=1.2, color="#444")
ax[0].set(xlabel="step", ylabel="InfoNCE loss", title="loss (not the thing we select on)")

ax[1].axhspan(0.48, 0.52, color="#bbb", alpha=.35, zorder=0)
ax[1].plot(H[:, 0], H[:, 2], lw=1.6, color="#1b6ca8", label="model (debiased)")
ax[1].plot(H[:, 0], H[:, 3], lw=1.2, color="#c1121f", ls="--", label="shuffled query")
ax[1].axhline(BAR, color="#e08b00", lw=1.4, ls=":", label=f"NN retrieval bar {BAR:.4f}")
ax[1].axhline(0.5, color="#888", lw=.8)
if best_step >= 0:
    ax[1].axvline(best_step, color="#1b6ca8", lw=.7, alpha=.5)
    ax[1].annotate(f"best {best:.4f}", (best_step, best), textcoords="offset points",
                   xytext=(4, 6), fontsize=8, color="#1b6ca8")
ax[1].set(xlabel="step", ylabel="debiased rank", title="held-out gate, with controls")
ax[1].legend(fontsize=7.5, loc="lower right")
plt.tight_layout(); plt.show()

## 8. Final gate + paired bootstrap

Per-row ranks are computed once against the full candidate set, then the
**paired** differences are bootstrapped. Resampling rows before ranking would
also shrink the candidate pool and confound the two effects.

In [ ]:
model.load_state_dict(torch.load("plife_best.pt")["model"]); model.eval()
g = gate(model)
print(f"{'variant':<34}{'cosine':>9}{'rank':>8}{'debiased':>10}")
for k, P in BASE.items():
    print(f"{k:<34}{cos_rows(P,Yv,Mv):>9.4f}{rank_raw(P,Yv,Mv):>8.4f}{rank_debiased(P,Yv,Mv):>10.4f}")
for k, v in g.items():
    print(f"{k:<34}{v[0]:>9.4f}{v[1]:>8.4f}{v[2]:>10.4f}")

def boot(d, reps=10000):
    d = d.detach().cpu().numpy()
    r = np.random.default_rng(0)
    m = d[r.integers(0, len(d), (reps, len(d)))].mean(1)
    return d.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5), float((m > 0).mean())

bar_rows = rank_debiased(BASE["NN retrieval on Feat  <-BAR"], Yv, Mv, per_row=True)
print(f"\nPAIRED BOOTSTRAP ({len(val_t)} val perturbations, 10k resamples)")
print(f"{'':<44}{'diff':>9}{'95% CI':>22}{'P(>0)':>8}")
for k in g:
    if k.startswith("query SHUF"):
        continue
    m, lo, hi, p = boot(g[k][3] - bar_rows)
    tag = "" if lo > 0 else ("  <- NOT SIGNIFICANT" if hi > 0 else "  <- WORSE THAN THE BAR")
    print(f"{k + '  vs  NN retrieval':<44}{m:>+9.4f}   [{lo:+.4f}, {hi:+.4f}]{p:>8.3f}{tag}")

print("\nCPU reference to beat: bilinear F=96 d=96 -> 0.5689 vs bar 0.5265 (+0.0424)")
print("Corpus-internal gate on held-out K562. Not a VCC score.")

### Results
Left: margin over the retrieval bar with 95% bootstrap CIs — a bar whose
interval crosses zero has not been shown to beat retrieval. Right: the
distribution of per-perturbation ranks. A model that is merely *average*-good
and one that nails a subset while failing the rest can share a mean, and only
the histogram tells them apart.

In [ ]:
names, mus, los, his = [], [], [], []
for k in g:
    if k.startswith("query SHUF"):
        continue
    m, lo, hi, _ = boot(g[k][3] - bar_rows)
    names.append(k); mus.append(m); los.append(m - lo); his.append(hi - m)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
cols = ["#1b6ca8" if l > 0 else "#c1121f" for l, m in zip(
    [m - e for m, e in zip(mus, los)], mus)]
ax[0].barh(names, mus, xerr=[los, his], color=cols, height=.55,
           error_kw=dict(lw=1.1, capsize=3, ecolor="#333"))
ax[0].axvline(0, color="#333", lw=1)
ax[0].set(xlabel="debiased rank  −  NN retrieval", title="margin over the bar (95% CI)")

edges = np.linspace(0, 1, 41)
ax[1].hist(bar_rows.cpu().numpy(), bins=edges, alpha=.55, label="NN retrieval",
           color="#e08b00")
ax[1].hist(g["FULL"][3].cpu().numpy(), bins=edges, alpha=.55, label="model",
           color="#1b6ca8")
ax[1].axvline(.5, color="#888", lw=.8, ls="--")
ax[1].set(xlabel="per-perturbation rank (0.5 = chance)", ylabel="count",
          title="who wins, and by how much")
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()